#                                                       SLEEP HEALTH AND LIFESTYLE DATASET

## IMPORT LIBRARIES

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import shapiro

 ## LOAD THE DATASET

In [ ]:
df=pd.read_csv("sleep_health_lifestyle_dataset.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'sleep_health_lifestyle_dataset.csv'

## PROBLEM STATEMENT

The purpose of this analysis is to investigate how various lifestyle factors (such as BMI, daily steps, alcohol consumption, stress level, and physical activity) influence sleep duration and overall sleep disorder status. The goal is to identify key predictors of poor sleep health and help design recommendations for better sleep hygiene.

## DATASET DETAILS

In [ ]:
#Get Info About Data Types and Non-Null Counts
df.info()

In [ ]:
#View First 5 Rows (Head)
df.head(5)

In [ ]:
#View Last 5 Rows (Tail)
df.tail(5)

In [ ]:
#Get Column Names
df.columns

In [ ]:
#Get Shape (Rows, Columns)
df.shape

In [ ]:
#Count of Unique Values in Each Column
df.nunique()

In [ ]:
#Summary Statistics for Numeric Columns
df.describe()

## CHECKING FOR MISSING VALUES

In [ ]:
missing_values=df.isnull()
print(missing_values)

In [ ]:
#To count missing values column-wise
missing_values=df.isnull().sum()
print(missing_values)

In [ ]:
#To check if any missing values exist at all
missing_values=df.isnull().values.any()
print(missing_values)

## Fill Missing Values in Sleep Disorder column

In [ ]:
#conforming how many values are missing
df['Sleep Disorder'].isnull().sum()

In [ ]:
#the best approach is to fill it with the mode (most frequent category)
df['Sleep Disorder'] = df['Sleep Disorder'].fillna(df['Sleep Disorder'].mode()[0])

In [ ]:
#After filling, verify that there are no missing values:
df['Sleep Disorder'].isnull().sum()

# INSIGHT - 1:
Missing values in the column Sleep Disorder have been filled using the mode, ensuring categorical consistency in the data.

Why We Use mode() for Categorical Columns ?
as mode = Most frequently occurring value in a column.
Example: In a column with values ["Insomnia", "None", "Insomnia", NaN], the mode is "Insomnia".

we can’t use mean or median because mean or median are not defined for strings like "Insomnia" or "Sleep Apnea". Trying to use them will throw an error.

## CHECKING FOR NON MISSING VALUES

In [ ]:
non_missing_values=df.notnull()
print(non_missing_values)

In [ ]:
#visiualizing non_missing_values using seaborn heatmap
sns.heatmap(non_missing_values.isnull(), cmap='viridis', cbar=False, yticklabels=False)
plt.title('Heatmap Showing Non-Missing Values')
plt.show()

## HANDLING NOISE DATA

In [ ]:
#Visual Inspection using Boxplots
import seaborn as sns
import matplotlib.pyplot as plt
columns_to_check = ['Age', 'Heart Rate (bpm)', 'Sleep Duration (hours)', 'Stress Level (scale: 1-10)', 'Daily Steps','Quality of Sleep (scale: 1-10)','Physical Activity Level (minutes/day)']

for col in columns_to_check:
    plt.figure(figsize=(6, 2))
    sns.boxplot(x=df[col])
    plt.title(f'Boxplot of {col}')
    plt.show()

In [ ]:
#Detect outliers with IQR for these numerical columns
numerical_cols = [
    'Age',
    'Sleep Duration (hours)',
    'Quality of Sleep (scale: 1-10)',
    'Physical Activity Level (minutes/day)',
    'Stress Level (scale: 1-10)',
    'Heart Rate (bpm)',
    'Daily Steps'
]

for col in numerical_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    print(f'Outliers detected in "{col}": {len(outliers)}')

In [ ]:
#Check for those 3 outlier values detected in Age column
Q1 = df['Age'].quantile(0.25)
Q3 = df['Age'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers_age = df[(df['Age'] < lower_bound) | (df['Age'] > upper_bound)]
print(outliers_age[['Person ID', 'Age']])


In [ ]:
#Cap outliers (replace them with lower or upper bound):
Q1 = df['Age'].quantile(0.25)
Q3 = df['Age'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Cap the values at the IQR bounds
df['Age'] = df['Age'].apply(lambda x: lower_bound if x < lower_bound else upper_bound if x > upper_bound else x)


In [ ]:
#After Capping — Re-Plot the Boxplot
plt.figure(figsize=(6, 4))
sns.boxplot(x=df['Age'])
plt.title('Boxplot of Age after Capping Outliers')
plt.show()

## INSIGHT - 2:
Only 3 records in the Age column were identified as statistical outliers. These were extreme values beyond the typical age range seen in the dataset. Instead of deleting them, they were capped using IQR thresholds to preserve the dataset's size and context

No significant outliers were observed in most other numerical columns, indicating well-behaved data with minimal noise.

## DATA PROCESSING

### 1. NORMALIZATION

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Select numerical columns to normalize
num_cols = ['Age', 'Sleep Duration (hours)', 'Quality of Sleep (scale: 1-10)',
            'Physical Activity Level (minutes/day)', 'Stress Level (scale: 1-10)',
            'Heart Rate (bpm)', 'Daily Steps']

# Create the scaler
scaler = MinMaxScaler()

# Apply normalization
df[num_cols] = scaler.fit_transform(df[num_cols])

# Preview
df[num_cols].head()


### INSIGHT:
Normalization was applied to the numerical features to scale their values between 0 and 1 using Min-Max scaling.
FORMULA USE:
Normalized Value=𝑥−min(𝑥)/max(𝑥)−min(𝑥)

### 2. ENCODING

In [ ]:
#Encoding of categorical column like gender, occupation, BMI category& sleep disorder
categorical_cols = ['Gender', 'Occupation', 'BMI Category', 'Sleep Disorder']

df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

In [ ]:
#encoding of categorical column : blood pressure

In [ ]:
df[['Systolic_BP', 'Diastolic_BP']] = df['Blood Pressure (systolic/diastolic)'].str.split('/', expand=True)
df['Systolic_BP'] = pd.to_numeric(df['Systolic_BP'])
df['Diastolic_BP'] = pd.to_numeric(df['Diastolic_BP'])
df.drop(columns=['Blood Pressure (systolic/diastolic)'], inplace=True)

### INSIGHT:
Here, I used One-Hot Encoding for categorical columns like Gender, Occupation, BMI Category, and Sleep Disorder. This method was chosen because these categories don’t have any specific order, and One-Hot Encoding helps convert them into numeric form without creating any confusion about their importance or rank.

I did not use Label Encoding because it gives numbers to categories that might suggest an order, which is not true for my data.

For the Blood Pressure column, I separated it into two numeric columns (Systolic and Diastolic) instead of encoding because these are continuous numerical values, not categories. Splitting keeps the important numeric information intact for better analysis.

### 3. DISCRITIZATION TECHNIQUE

In [ ]:
# discretization on the categorical columns in my dataset

In [ ]:
import pandas as pd

# Age Grouping
bins_age = [0, 30, 60, 100]
labels_age = ['Young', 'Adult', 'Senior']
df['Age_Group'] = pd.cut(df['Age'] * 100, bins=bins_age, labels=labels_age, include_lowest=True)

# Sleep Duration Grouping
bins_sleep = [0, 5, 8, 24]
labels_sleep = ['Short Sleep', 'Normal Sleep', 'Long Sleep']
df['Sleep_Duration_Group'] = pd.cut(df['Sleep Duration (hours)'] * 24, bins=bins_sleep, labels=labels_sleep, include_lowest=True)

# Stress Level Grouping
bins_stress = [0, 3, 6, 10]
labels_stress = ['Low', 'Medium', 'High']
df['Stress_Level_Group'] = pd.cut(df['Stress Level (scale: 1-10)'] * 10, bins=bins_stress, labels=labels_stress, include_lowest=True)

# Physical Activity Level Grouping
bins_activity = [0, 30, 60, 1440]  # 1440 = max minutes/day
labels_activity = ['Sedentary', 'Moderate', 'Active']
df['Activity_Level_Group'] = pd.cut(df['Physical Activity Level (minutes/day)'] * 1440, bins=bins_activity, labels=labels_activity, include_lowest=True)


In [ ]:
# Categories for Systolic BP
bins_sys = [0, 120, 129, 139, 180]
labels_sys = ['Normal', 'Elevated', 'Hypertension Stage 1', 'Hypertension Stage 2']

df['Systolic_BP_Category'] = pd.cut(df['Systolic_BP'], bins=bins_sys, labels=labels_sys, include_lowest=True)

# Categories for Diastolic BP
bins_dia = [0, 80, 89, 180]
labels_dia = ['Normal', 'Hypertension Stage 1', 'Hypertension Stage 2']

df['Diastolic_BP_Category'] = pd.cut(df['Diastolic_BP'], bins=bins_dia, labels=labels_dia, include_lowest=True)

In [ ]:
print(df['Age_Group'].value_counts(dropna=False))
print(df['Sleep_Duration_Group'].value_counts(dropna=False))
print(df['Stress_Level_Group'].value_counts(dropna=False))
print(df['Activity_Level_Group'].value_counts(dropna=False))

In [ ]:
print(df['Systolic_BP_Category'].value_counts(dropna=False))
print(df['Diastolic_BP_Category'].value_counts(dropna=False))

### INSIGHT
I used a technique called Discretization to make continuous data easier to understand and analyze.
Discretization means dividing continuous numerical values into fixed categories or groups.
For example, instead of working with many different age values like 21, 35, 47, etc., I grouped them into:
Young (0–30 years)
Adult (31–60 years)
Senior (61+ years)
This helps in reducing the complexity of data and makes patterns easier to spot.

I applied discretization on the following columns in my dataset:
Age → grouped into 'Young', 'Adult', 'Senior'
Sleep Duration → grouped into 'Short Sleep', 'Normal Sleep', 'Long Sleep'
Stress Level → grouped into 'Low', 'Medium', 'High'
Physical Activity Level → grouped into 'Sedentary', 'Moderate', 'Active'
Blood Pressure (Systolic & Diastolic) → grouped into medical categories like 'Normal', 'Elevated', 'Hypertension Stage 1/2'

Discretization helped me:
Understand the data more easily
Analyze behavior and health patterns across different age or activity groups
Prepare the data for visualization and interpretation

## Exploratory Data Analysis (EDA)

### 1.Summary Statistics (Central Tendency & Dispersion)

In [ ]:
# Summary statistics for numerical columns
summary_stats = df.describe()
print(summary_stats)

#### INSIGHT-1:
Central Tendency & Dispersion
From the mean and std, we can see which values are spread out more.
Example: If Stress Level has a high standard deviation, it means people have a wide range of stress.
The min and max values show if there are any extreme outliers.
We can use median (50%) from .describe() to compare with mean — if they differ a lot, the data may be skewed.

### 2.Checking Normality

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

columns_to_plot = ['Age', 'Sleep Duration (hours)', 'Stress Level (scale: 1-10)',
                   'Physical Activity Level (minutes/day)', 'Heart Rate (bpm)']

for col in columns_to_plot:
    plt.figure(figsize=(6, 4))
    sns.histplot(df[col], kde=True, color='skyblue')
    plt.title(f'Distribution of {col}')
    plt.show()

# Check skewness
print(df[columns_to_plot].skew())


#### INSIGHT-2:
| Column                                    | Skewness  | Interpretation                                                        |
| ----------------------------------------- | --------- | --------------------------------------------------------------------- |
| **Age**                                   | `0.3579`  | Slightly **positively skewed** (a bit more values on the lower side). |
| **Sleep Duration (hours)**                | `-0.0638` | Almost **symmetrical** (very close to normal).                        |
| **Stress Level (scale: 1-10)**            | `-0.0206` | Very close to **normal distribution**.                                |
| **Physical Activity Level (minutes/day)** | `0.0380`  | Nearly **normally distributed**, slight positive skew.                |
| **Heart Rate (bpm)**                      | `-0.0851` | Almost **normal**, very mild negative skew.                           |

All your values are between -0.5 and +0.5, which indicates the distributions are approximately normal.


### 3.Correlation Analysis

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(14, 10))
corr = df.corr(numeric_only=True)
sns.heatmap(corr, annot=True, fmt=".2f", cmap='viridis', square=True)

plt.title("Correlation Matrix")
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

### INSIGHT-3:
A correlation matrix is used to measure the linear relationship between numerical variables in the dataset. The correlation values range from –1 to +1, where:
->+1 indicates a perfect positive correlation
->–1 indicates a perfect negative correlation
->0 indicates no linear correlation

Based on the matrix:
| **S.No** | **Variable Pair**                         | **Correlation** | **Insight**                                                               |
| -------- | ----------------------------------------- | --------------- | ------------------------------------------------------------------------- |
| 1        | Age vs Systolic BP                        | 0.85            | Strong positive correlation. As age increases, systolic BP tends to rise. |
| 2        | Age vs Diastolic BP                       | 0.73            | Strong correlation. Diastolic BP increases with age.                      |
| 3        | Systolic BP vs Diastolic BP               | 0.86            | Very strong correlation. Both pressures tend to rise together.            |
| 4        | BMI Obese vs Systolic BP                  | 0.52            | Obese individuals tend to have higher systolic BP.                        |
| 5        | BMI Obese vs Sleep Disorder (Sleep Apnea) | \~0.20–0.30     | Moderate correlation. Obesity may contribute to sleep disorders.          |
| 6        | Stress Level vs Quality of Sleep          | \~–0.20         | Weak negative correlation. Higher stress may reduce sleep quality.        |
| 7        | Age vs Heart Rate                         | \~0.00          | No significant correlation. Heart rate is not dependent on age here.      |
| 8        | Physical Activity vs Daily Steps          | \~0.30          | Moderate correlation. Active people usually take more daily steps.        |


## Data Visualization

### 1. HISTOGRAMS

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

columns = df.columns.tolist()

plt.figure(figsize=(15, 20))

for i, col in enumerate(columns, 1):
    plt.subplot(6, 3, i)
    sns.histplot(df[col], bins=30, kde=True, color='brown')
    plt.title(f'Distribution of {col}')
    plt.xlabel(col)
    plt.ylabel('Frequency')

plt.tight_layout()
plt.show()


In [ ]:
print(f"Total columns: {len(df.columns)}")
print(df.columns.tolist())

### 2.BAR PLOTS

In [ ]:
# Count of people by Gender
import matplotlib.pyplot as plt
import seaborn as sns

gender_counts = df['Gender_Male'].value_counts().rename(index={0: 'Female', 1: 'Male'})

plt.figure(figsize=(6,4))
sns.barplot(x=gender_counts.index, y=gender_counts.values)
plt.title('Count of People by Gender')
plt.xlabel('Gender')
plt.ylabel('Count')
plt.show()


In [ ]:
#Count of people with Sleep Apnea
sleep_apnea_counts = df['Sleep Disorder_Sleep Apnea'].value_counts().rename(index={0: 'No Sleep Apnea', 1: 'Sleep Apnea'})

plt.figure(figsize=(6,4))
sns.barplot(x=sleep_apnea_counts.index, y=sleep_apnea_counts.values)
plt.title('Count of Peoples by Sleep Apnea Status')
plt.xlabel('Sleep Apnea Status')
plt.ylabel('Count')
plt.show()

In [ ]:
#Average Sleep Duration by BMI Categories
import numpy as np

# Create a new column for BMI category by checking which flag is 1
def bmi_category(row):
    if row['BMI Category_Obese'] == 1:
        return 'Obese'
    elif row['BMI Category_Overweight'] == 1:
        return 'Overweight'
    elif row['BMI Category_Underweight'] == 1:
        return 'Underweight'
    else:
        return 'Normal/Other'

df['BMI_Category'] = df.apply(bmi_category, axis=1)

plt.figure(figsize=(8,5))
sns.barplot(x='BMI_Category', y='Sleep Duration (hours)', data=df, order=['Underweight', 'Normal/Other', 'Overweight', 'Obese'])
plt.title('Average Sleep Duration by BMI Category')
plt.xlabel('BMI Category')
plt.ylabel('Average Sleep Duration (hours)')
plt.show()


In [ ]:
#Average Quality of Sleep by Occupation
def occupation_category(row):
    if row['Occupation_Office Worker'] == 1:
        return 'Office Worker'
    elif row['Occupation_Retired'] == 1:
        return 'Retired'
    elif row['Occupation_Student'] == 1:
        return 'Student'
    else:
        return 'Other'

df['Occupation'] = df.apply(occupation_category, axis=1)

plt.figure(figsize=(8,5))
sns.barplot(x='Occupation', y='Quality of Sleep (scale: 1-10)', data=df)
plt.title('Average Quality of Sleep by Occupation')
plt.xlabel('Occupation')
plt.ylabel('Average Sleep Quality')
plt.show()

### 3.SCATTER PLOT

In [ ]:
#Stress Level vs Physical Activity Level
plt.figure(figsize=(8,6))
sns.scatterplot(
    x='Physical Activity Level (minutes/day)',
    y='Stress Level (scale: 1-10)',
    hue='Gender',
    data=df,
    palette='Set1'
)

plt.title('Stress Level vs Physical Activity Level')
plt.xlabel('Physical Activity Level (minutes/day)')
plt.ylabel('Stress Level (scale: 1-10)')
plt.legend(title='Gender')
plt.show()


## 4. BOX PLOT

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt


def get_occupation(row):
    if row['Occupation_Office Worker'] == 1:
        return 'Office Worker'
    elif row['Occupation_Retired'] == 1:
        return 'Retired'
    elif row['Occupation_Student'] == 1:
        return 'Student'
    else:
        return 'Other'

df['Occupation'] = df.apply(get_occupation, axis=1)

plt.figure(figsize=(10,6))
sns.boxplot(x='Occupation', y='Sleep Duration (hours)', data=df, palette='pastel')
plt.title('Sleep Duration Distribution by Occupation')
plt.xlabel('Occupation')
plt.ylabel('Sleep Duration (hours)')
plt.show()


## 5. HEAT MAP

In [ ]:
#Sleep Duration vs Stress Level
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

df['Sleep_Bin'] = pd.cut(df['Sleep Duration (hours)'], bins=6)
df['Stress_Bin'] = pd.cut(df['Stress Level (scale: 1-10)'], bins=5)


heatmap_data = df.pivot_table(index='Sleep_Bin', columns='Stress_Bin', aggfunc='size', fill_value=0)

plt.figure(figsize=(10, 6))
sns.heatmap(heatmap_data, annot=True, fmt='d', cmap='YlOrRd')
plt.title('Heatmap of Sleep Duration vs Stress Level')
plt.xlabel('Stress Level (Binned)')
plt.ylabel('Sleep Duration (Binned)')
plt.show()


In [ ]:
#Heatmap of Occupation vs BMI Category
# Step 1: Recreate categorical labels
def get_bmi(row):
    if row['BMI Category_Obese'] == 1:
        return 'Obese'
    elif row['BMI Category_Overweight'] == 1:
        return 'Overweight'
    elif row['BMI Category_Underweight'] == 1:
        return 'Underweight'
    else:
        return 'Normal/Other'

def get_occupation(row):
    if row['Occupation_Office Worker'] == 1:
        return 'Office Worker'
    elif row['Occupation_Retired'] == 1:
        return 'Retired'
    elif row['Occupation_Student'] == 1:
        return 'Student'
    else:
        return 'Other'

df['BMI_Category'] = df.apply(get_bmi, axis=1)
df['Occupation'] = df.apply(get_occupation, axis=1)

# Step 2: Pivot tabl
heatmap_data2 = df.pivot_table(index='Occupation', columns='BMI_Category', aggfunc='size', fill_value=0)

# Step 3: Heatmap
plt.figure(figsize=(8,6))
sns.heatmap(heatmap_data2, annot=True, fmt='d', cmap='Blues')
plt.title('Heatmap of Occupation vs BMI Category')
plt.xlabel('BMI Category')
plt.ylabel('Occupation')
plt.show()


## 6. PAIR PLOT

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.pairplot(df[numeric_cols], diag_kind='kde', plot_kws={'alpha':0.5, 's':30, 'edgecolor':'k'}, height=2.5)
plt.suptitle('Pair Plot of Numeric Variables', y=1.02)
plt.show()


## CONCLUSION

In this analysis, we studied how different lifestyle and health factors affect sleep duration and sleep disorders. We worked with the Sleep Health and Lifestyle dataset to find important patterns and relationships.

During data cleaning, only in the Age column was found to have some missing values. These missing ages were handled carefully by capping the extreme outliers instead of removing them to keep the dataset complete. We also scaled the numerical data to a common range to make comparisons easier.

We converted all non-numeric data, like gender and occupation, into numbers using one-hot encoding because these categories do not have any natural order. Also, we grouped continuous data like age, sleep duration, and stress into categories to better understand patterns across different groups.

Our analysis showed that the data is mostly well-behaved, with most variables having a normal-like distribution. We found some strong relationships too. For example, as people get older, their blood pressure tends to rise, and obese individuals are more likely to have higher blood pressure and sleep apnea. We also saw that stress slightly lowers sleep quality, and people who are more physically active tend to take more daily steps.

Overall, these insights suggest that managing factors like obesity, stress, and physical activity can play an important role in improving sleep health. Future work could explore how to use these factors to design better sleep recommendations and promote healthier lifestyles.

